# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields as recommended for robust and reproducible workflows.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Each record set, field, and column is referenced only by its `@id` as per the Croissant standard.

In [ ]:
# List all record sets and their fields/columns using @id
record_sets = dataset.metadata.recordSet
if record_sets is None or len(record_sets) == 0:
    print('No record sets found in the metadata!')
else:
    for rs in record_sets:
        print(f"\nRecordSet: {rs['@id']}")
        # Print associated fields and columns
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            print(f"    - {field['@id']}")
        # Print columns
        columns = rs.get('column', [])
        if not isinstance(columns, list):
            columns = [columns]
        print("  Columns:")
        for col in columns:
            print(f"    - {col['@id']}")

For quick exploration, let's print a sample record from each available record set (using the set's `@id`).

In [ ]:
# Print one example record for each record set (by @id)
if record_sets:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f'First record from record set {rs_id}:')
        try:
            iterator = dataset.records(record_set=rs_id)
            rec = next(iterator)
            print(rec)
        except Exception as e:
            print(f'  [!] Could not load records: {e}')
        print('-'*60)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use record set and field `@id`s from the overview above.

> _**Note**: The rest of this notebook uses real `@id` values from the loaded metadata, and all references to fields/columns use their `@id`._

In [ ]:
# Choose record set(s) to extract
# If you know the record set(s) you wish to analyze, plug in their @id(s) here:
# (In this dataset, you may want to inspect the main protein data table record set)
selected_record_set_ids = []
for rs in (record_sets or []):
    selected_record_set_ids.append(rs['@id'])
if not selected_record_set_ids:
    print('No record sets detected, cannot extract records.')

dataframes = {}
for record_set_id in selected_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Data from record set {record_set_id} (shape: {df.shape}) loaded. Columns (by @id):\n{list(df.columns)}\n")
    else:
        print(f"No records found for record set {record_set_id}.")

In [ ]:
# Display first rows from the main data table (if present)
if selected_record_set_ids:
    main_rs_id = selected_record_set_ids[0]
    if main_rs_id in dataframes:
        print(f"First 5 records in record set {main_rs_id}:")
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform common data processing: filtering records by criteria, normalizing numeric fields, and grouping data.

All field names reference the dataset's Croissant `@id`s.

In [ ]:
# Choose a numeric field (by @id), e.g., 'cr:PeptideCount'
import numpy as np

# Try to detect a numeric column in the first DataFrame
main_rs_id = selected_record_set_ids[0] if selected_record_set_ids else None
numeric_field_id = None
if main_rs_id:
    df = dataframes[main_rs_id]
    # Pick the first int/float column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print(f"No numeric columns found in {main_rs_id}. Please check dataset schema.")
else:
    print('No main record set detected.')

# Set a threshold for filtering
threshold = None
if numeric_field_id and main_rs_id:
    # Compute a threshold such as the column median
    column = df[numeric_field_id]
    threshold = np.nanmedian(column)
    # Filter records
    filtered_df = df[column > threshold].copy()
    print(f"Filtered records in '{main_rs_id}' where '{numeric_field_id}' > {threshold} (found {len(filtered_df)} records):\n")
    display(filtered_df.head())

    # Normalize the numeric field
    field_norm = f"{numeric_field_id}_normalized"
    filtered_df[field_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records (first few):")
    display(filtered_df[[numeric_field_id, field_norm]].head())

    # Try grouping by another field (pick first string column)
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field_id:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
        print(f"Grouped data by '{group_field}' (showing mean of '{numeric_field_id}'):")
        display(grouped_df.head())
    else:
        print("No grouping field available for demonstration.")
else:
    print("Cannot perform EDA: No numeric fields found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or another plotting library. All fields are referenced by their Croissant `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field
if numeric_field_id and main_rs_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set '{main_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If there is a grouping field, show mean bar plot
    if group_field:
        plt.figure(figsize=(10, 4))
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        group_means.plot(kind='bar')
        plt.title(f"Mean '{numeric_field_id}' per '{group_field}' in '{main_rs_id}'")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrates how to load and explore a Croissant-standard dataset using the `mlcroissant` API—referencing all record sets, fields, and columns by their unique `@id`s for interoperability and reproducibility.

Key findings from this exploration (update with specific dataset results):
- The dataset contains mass spectrometry-based protein quantification tables from human extracellular vesicle samples.
- Numeric field distributions and simple aggregations are easily accessible by `@id`, facilitating both automation and benchmarking.

Use this workflow on other Croissant-conformant datasets by updating the Croissant schema URL and adjusting field/column `@id` as needed.